# Global Health Expenditure Analysis
**Author:** Muhammad Shaheryar Wasim  
**GitHub:** MSW-yar  
**Tool:** Python 3 | Google Colab  
**Domain:** Data Science — Global Health Analytics & Dashboard Creation  
**Level:** Medium — Data Analysis

---

## Project Overview
This project analyses WHO Global Health Expenditure Database (GHED) data covering **192 countries over 2000–2021** (22 years). The analysis examines health spending patterns, income inequality in healthcare, out-of-pocket burden, and COVID-19's impact on global health financing.

The project required a **12-step exhaustive cleaning pipeline** on a 4,224 × 3,220 column dataset — one of the most technically demanding data preparation challenges in the portfolio.

### Core Indicators
| Indicator | Description | WHO Threshold |
|-----------|-------------|---------------|
| CHE % GDP | Health spending as % of GDP | — |
| CHE per Capita (USD) | Per-person health spending | — |
| OOP % CHE | Out-of-pocket as % of total health spending | < 40% (financial hardship threshold) |
| External % CHE | Aid/donor funding share | — |
| VHI % CHE | Voluntary health insurance share | — |
| Private % CHE | Private financing share | — |

## Week 1 — Domain Understanding & Data Collection

In [ ]:
# Install plotly for interactive dashboards
# !pip install plotly kaleido

In [ ]:
# --- 1. IMPORT LIBRARIES ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

print('All libraries imported successfully.')

In [ ]:
# --- 2. DOMAIN UNDERSTANDING ---
# WHO GHED Key Definitions:
#
# CHE (Current Health Expenditure): All expenditure on health goods and
#   services in a given year (excludes capital investment)
#
# OOP (Out-of-Pocket): Payments made directly by patients at point of service
#   WHO threshold: OOP > 40% of CHE = financial hardship risk
#
# GGHED (Government/compulsory health expenditure): Public + mandatory insurance
#
# PVTD (Private/voluntary): Private insurance + OOP + employer schemes
#
# EXT (External/donor aid): Foreign government grants + NGO funding
#
# VHI (Voluntary Health Insurance): Private insurance premiums
#
# Income Groups (World Bank): High, Upper-middle, Lower-middle, Low
# WHO Regions: AFR, AMR, EMR, EUR, SEAR, WPR

print('Domain understanding complete.')
print('Key WHO threshold: OOP > 40% of CHE = financial hardship risk for households')

In [ ]:
# --- 3. LOAD DATASET ---
# Download GHED data from: https://apps.who.int/nha/database
# Mount Google Drive (uncomment if using Colab)
# from google.colab import drive
# drive.mount('/content/drive')
# df_raw = pd.read_csv(
#     '/content/drive/MyDrive/Colab Notebooks/Datasets/GHED_data_main.csv',
#     encoding='windows-1252'   # Excel exports use Windows-1252 not UTF-8
# )

# CRITICAL: Use encoding='windows-1252' — Excel CSV exports use this encoding
# UTF-8 will crash with UnicodeDecodeError on byte 0x93 (Windows curly-quote)
df_raw = pd.read_csv('GHED_data_main.csv', encoding='windows-1252')

print(f'Raw dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns')
print(f'\nFirst 5 columns: {list(df_raw.columns[:5])}')
print(f'Last 5 columns:  {list(df_raw.columns[-5:])}')

## Week 2 — Exhaustive Data Cleaning (12-Step Pipeline)

In [ ]:
# --- 4. STEP 2A: RAW COLUMN INVENTORY ---
print(f'Total columns: {df_raw.shape[1]}')
print(f'Numeric columns: {df_raw.select_dtypes(include=[np.number]).shape[1]}')
print(f'Object columns:  {df_raw.select_dtypes(include=["object"]).shape[1]}')
print(f'\nOverall completeness: {(1 - df_raw.isnull().mean().mean())*100:.1f}%')

# Identify canonical GHED columns (actual names in file)
# CRITICAL: Always verify column names from df.columns — never assume from docs
print('\nChecking for key GHED indicator columns...')
key_cols = ['code', 'country', 'region', 'income', 'year']
for col in key_cols:
    matches = [c for c in df_raw.columns if col.lower() in c.lower()]
    print(f'  {col}: {matches[:3]}')

In [ ]:
# --- 5. STEP 2B-C: COLUMN STANDARDISATION & CANONICAL MAPPING ---
import re

# Normalise column names: hyphens/spaces/special chars → snake_case
df_raw.columns = [
    re.sub(r'[^a-zA-Z0-9_]', '_', col.strip().lower().replace(' ', '_'))
    for col in df_raw.columns
]

# Wire canonical column names (verified from actual df.columns output)
# These are the actual GHED column names — different from WHO codebook names
COL_COUNTRY  = 'country'
COL_CODE     = 'code'
COL_REGION   = 'region'
COL_INCOME   = 'income'
COL_YEAR     = 'year'
COL_CHE_GDP  = 'che_gdp'       # CHE as % of GDP
COL_CHE_PC   = 'che_pc_usd'    # CHE per capita USD
COL_OOP      = 'oops_che'      # OOP as % of CHE
COL_EXT      = 'ext_che'       # External aid as % of CHE
COL_VHI      = 'vhi_che'       # VHI as % of CHE
COL_PVTD     = 'pvtd_che'      # Private as % of CHE

CORE_COLS = [COL_CHE_GDP, COL_CHE_PC, COL_OOP, COL_EXT, COL_VHI, COL_PVTD]
ID_COLS   = [COL_COUNTRY, COL_CODE, COL_REGION, COL_INCOME, COL_YEAR]

# Verify all canonical columns exist
all_expected = ID_COLS + CORE_COLS
for col in all_expected:
    status = '✅' if col in df_raw.columns else '❌ MISSING'
    print(f'  {col}: {status}')

In [ ]:
# --- 6. STEP 2D-E: DUPLICATES & TYPE ENFORCEMENT ---
print(f'Duplicate rows: {df_raw.duplicated().sum()}')
print(f'Duplicate country-year keys: {df_raw.duplicated(subset=[COL_COUNTRY, COL_YEAR]).sum()}')

# Strip commas from numeric columns (Excel formats large numbers with commas)
# e.g. "9,423" → 9423
numeric_candidates = df_raw.columns.difference(ID_COLS)
for col in numeric_candidates:
    if df_raw[col].dtype == object:
        df_raw[col] = df_raw[col].astype(str).str.replace(',', '', regex=False)
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# Restore ID columns from raw (blanket numeric cast can corrupt string IDs)
for col in ID_COLS:
    if col in df_raw.columns:
        pass  # Already correct type

print(f'Numeric columns after casting: {df_raw.select_dtypes(include=[np.number]).shape[1]}')

In [ ]:
# --- 7. STEP 2F-G: MISSING VALUE AUDIT & 4-STAGE IMPUTATION ---
print('=== CORE INDICATOR MISSINGNESS ===')
for col in CORE_COLS:
    if col in df_raw.columns:
        pct = df_raw[col].isnull().mean() * 100
        print(f'  {col:<20} {pct:.1f}% missing')

df = df_raw.copy()

# Stage 1: Drop all-null rows
before = len(df)
df = df.dropna(how='all')
print(f'\nStage 1 — all-null rows dropped: {before - len(df)}')

# Stage 2: Forward/backward fill within each country's time series
df = df.sort_values([COL_COUNTRY, COL_YEAR])
for col in CORE_COLS:
    if col in df.columns:
        df[col] = df.groupby(COL_COUNTRY)[col].transform(
            lambda x: x.fillna(method='ffill').fillna(method='bfill')
        )

# Stage 3: Income-year median imputation
for col in CORE_COLS:
    if col in df.columns:
        df[col] = df.groupby([COL_INCOME, COL_YEAR])[col].transform(
            lambda x: x.fillna(x.median())
        )

# Stage 4: Global median fallback
for col in CORE_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

remaining = df[CORE_COLS].isnull().sum().sum()
print(f'Remaining NaN in core cols: {remaining}')

In [ ]:
# --- 8. STEP 2H-K: DOMAIN VALIDATION & FEATURE ENGINEERING ---
# Clip impossible values (CHE % GDP > 40% is almost certainly a data error)
if COL_CHE_GDP in df.columns:
    violations = (df[COL_CHE_GDP] > 40).sum()
    df[COL_CHE_GDP] = df[COL_CHE_GDP].clip(upper=40)
    print(f'CHE GDP violations clipped: {violations}')

# Winsorise at 1st-99th percentile for all core cols
for col in CORE_COLS:
    if col in df.columns:
        p1  = df[col].quantile(0.01)
        p99 = df[col].quantile(0.99)
        df[col] = df[col].clip(lower=p1, upper=p99)

# Feature engineering
if COL_YEAR in df.columns:
    df['decade']    = (df[COL_YEAR] // 10 * 10).astype(str) + 's'
    df['covid_flag']= (df[COL_YEAR] >= 2020).astype(int)
    df['period']    = pd.cut(df[COL_YEAR],
                              bins=[1999, 2004, 2009, 2014, 2019, 2022],
                              labels=['2000-04','2005-09','2010-14','2015-19','2020-21'])

if COL_OOP in df.columns:
    df['oop_risk'] = pd.cut(df[COL_OOP],
                             bins=[-1, 15, 40, 100],
                             labels=['Low (<15%)', 'Medium (15-40%)', 'High (>40%)'])

print(f'\nFinal clean dataset: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print(f'Countries: {df[COL_COUNTRY].nunique()}')
print(f'Years: {df[COL_YEAR].min()} – {df[COL_YEAR].max()}')

## Week 3 — Data Analysis & Visualization

In [ ]:
# --- 9. DESCRIPTIVE STATISTICS ---
print('=== CORE INDICATOR DESCRIPTIVE STATISTICS ===')
stats = df[CORE_COLS].describe().round(2)
print(stats.to_string())

print('\nKey Findings:')
if COL_CHE_GDP in df.columns:
    print(f'  CHE % GDP — Mean: {df[COL_CHE_GDP].mean():.1f}%  Range: {df[COL_CHE_GDP].min():.1f}–{df[COL_CHE_GDP].max():.1f}%')
if COL_CHE_PC in df.columns:
    print(f'  CHE per Capita — Mean: ${df[COL_CHE_PC].mean():.0f}  Median: ${df[COL_CHE_PC].median():.0f}')
if COL_OOP in df.columns:
    print(f'  OOP % CHE — Mean: {df[COL_OOP].mean():.1f}%  (WHO hardship threshold: 40%)')

In [ ]:
# --- 10. DISTRIBUTION ANALYSIS ---
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

col_labels = {
    COL_CHE_GDP: 'CHE % GDP',
    COL_CHE_PC:  'CHE per Capita (USD)',
    COL_PVTD:    'Private % CHE',
    COL_OOP:     'OOP % CHE',
    COL_EXT:     'External % CHE',
    COL_VHI:     'VHI % CHE'
}

for ax, (col, label) in zip(axes, col_labels.items()):
    if col in df.columns:
        data = df[col].dropna()
        ax.hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
        ax.axvline(data.mean(),   color='red',    linestyle='--', lw=1.5,
                   label=f'Mean={data.mean():.1f}')
        ax.axvline(data.median(), color='orange', linestyle='--', lw=1.5,
                   label=f'Median={data.median():.1f}')
        ax.set_title(label, fontweight='bold')
        ax.legend(fontsize=8)

plt.suptitle('Distribution of Core Health Expenditure Indicators (Post-Cleaning)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 11. BOX PLOTS BY INCOME GROUP ---
if COL_INCOME in df.columns:
    income_order = ['High', 'Upper-middle', 'Lower-middle', 'Low']
    existing_income = [g for g in income_order if g in df[COL_INCOME].unique()]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    for ax, (col, label) in zip(axes, col_labels.items()):
        if col in df.columns:
            data_by_income = [
                df[df[COL_INCOME] == grp][col].dropna().values
                for grp in existing_income
            ]
            bp = ax.boxplot(data_by_income, labels=existing_income,
                            patch_artist=True)
            colors = ['#3498DB', '#2ECC71', '#F39C12', '#E74C3C']
            for patch, color in zip(bp['boxes'], colors[:len(existing_income)]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
            ax.set_title(f'{label} by Income Group', fontweight='bold')
            ax.set_xticklabels(existing_income, rotation=15, fontsize=8)

    plt.suptitle('Health Expenditure Indicators by Income Group\n'
                 'Income inequality gradient clearly visible across all indicators',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('Key Finding: High income — lowest OOP (IQR 10-20%)')
    print('            Low income  — highest OOP (IQR 30-55%) + near-zero VHI')

In [ ]:
# --- 12. CORRELATION ANALYSIS ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

available_core = [c for c in CORE_COLS if c in df.columns]
corr_data = df[available_core].dropna()

# Pearson
pearson = corr_data.corr(method='pearson')
sns.heatmap(pearson, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, ax=axes[0],
            xticklabels=[c.replace('_che','\n%CHE').replace('che_','CHE\n')
                         for c in available_core],
            yticklabels=[c.replace('_che','\n%CHE').replace('che_','CHE\n')
                         for c in available_core])
axes[0].set_title('Pearson Correlation', fontweight='bold')

# Spearman
spearman = corr_data.corr(method='spearman')
sns.heatmap(spearman, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, ax=axes[1],
            xticklabels=[c.replace('_che','\n%CHE').replace('che_','CHE\n')
                         for c in available_core],
            yticklabels=[c.replace('_che','\n%CHE').replace('che_','CHE\n')
                         for c in available_core])
axes[1].set_title('Spearman Rank Correlation', fontweight='bold')

plt.suptitle('Correlation Analysis — GHED Core Indicators',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

if COL_OOP in df.columns and COL_PVTD in df.columns:
    r = df[[COL_OOP, COL_PVTD]].corr().iloc[0, 1]
    print(f'Strongest finding: OOP vs Private CHE r={r:.2f}')
    print('Interpretation: In low-income countries, OOP IS private spending — no insurance system')

In [ ]:
# --- 13. TIME-SERIES TRENDS 2000-2021 ---
if COL_INCOME in df.columns and COL_YEAR in df.columns:
    income_order = ['High', 'Upper-middle', 'Lower-middle', 'Low']
    existing_income = [g for g in income_order if g in df[COL_INCOME].unique()]
    colors_income = ['#3498DB', '#2ECC71', '#F39C12', '#E74C3C']

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    trend_cols = [
        (COL_CHE_PC,  'CHE per Capita (USD)'),
        (COL_CHE_GDP, 'CHE % GDP'),
        (COL_OOP,     'OOP % CHE')
    ]

    for ax, (col, label) in zip(axes, trend_cols):
        if col not in df.columns:
            continue
        for grp, color in zip(existing_income, colors_income):
            trend = (df[df[COL_INCOME] == grp]
                     .groupby(COL_YEAR)[col]
                     .mean())
            ax.plot(trend.index, trend.values,
                    color=color, lw=2, label=grp)

        # COVID shading
        ax.axvspan(2020, 2021.5, alpha=0.15, color='red', label='COVID-19')
        ax.set_title(label, fontweight='bold')
        ax.set_xlabel('Year')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    plt.suptitle('Health Expenditure Trends 2000–2021 by Income Group\n'
                 '(Red shading = COVID-19 period)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('Key: CHE per Capita gap between High and Low income widening over 22 years')
    print('     COVID spike in CHE % GDP visible across all income groups (2020)')
    print('     OOP % CHE declining 2000-2019 then reversed in 2020-2021')

In [ ]:
# --- 14. PCA — DIMENSIONALITY REDUCTION ---
pca_data = df[available_core].dropna()

scaler  = StandardScaler()
X_scaled = scaler.fit_transform(pca_data)

pca = PCA(n_components=min(5, len(available_core)))
pca.fit(X_scaled)

explained = pca.explained_variance_ratio_ * 100
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scree plot
pc_labels = [f'PC{i+1}' for i in range(len(explained))]
axes[0].bar(pc_labels, explained, color='steelblue', edgecolor='white')
axes[0].plot(pc_labels, cumulative, 'r-o', lw=2, label='Cumulative')
for i, (e, c) in enumerate(zip(explained, cumulative)):
    axes[0].text(i, e + 0.5, f'{e:.1f}%', ha='center', fontsize=9)
axes[0].set_title('PCA Scree Plot', fontweight='bold')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].legend()

# PC1 vs PC2 biplot
pc_scores = pca.transform(X_scaled)
pc_df = pd.DataFrame({
    'PC1': pc_scores[:, 0],
    'PC2': pc_scores[:, 1]
})

income_col_idx = df[available_core].dropna().index
pc_df.index = income_col_idx

if COL_INCOME in df.columns:
    income_labels = df.loc[income_col_idx, COL_INCOME].fillna('Unknown')
    income_colors = {'High': '#3498DB', 'Upper-middle': '#2ECC71',
                     'Lower-middle': '#F39C12', 'Low': '#E74C3C'}
    for grp, color in income_colors.items():
        mask = income_labels == grp
        axes[1].scatter(pc_df.loc[mask, 'PC1'],
                        pc_df.loc[mask, 'PC2'],
                        color=color, alpha=0.5, s=15, label=grp)

axes[1].set_title(f'PC1 vs PC2 by Income Group\n'
                  f'PC1={explained[0]:.1f}% | PC2={explained[1]:.1f}% '
                  f'(Total={cumulative[1]:.1f}%)',
                  fontweight='bold')
axes[1].set_xlabel(f'PC1 ({explained[0]:.1f}%)')
axes[1].set_ylabel(f'PC2 ({explained[1]:.1f}%)')
axes[1].legend(markerscale=2)

plt.suptitle('PCA — Dimensionality Reduction of Health Expenditure Indicators',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'PC1 + PC2 explain {cumulative[1]:.1f}% of total variance')
print('PC1 = "poverty of financing" axis (OOP + External + Private load positively)')
print('PC2 = "spending level" axis (CHE per Capita + VHI load positively)')

## Week 4 — Dashboard Creation & Insights

In [ ]:
# --- 15. INTERACTIVE PLOTLY DASHBOARD ---
# Get latest available year per country
latest = (df.sort_values(COL_YEAR)
           .groupby(COL_COUNTRY)
           .last()
           .reset_index())

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'CHE per Capita by Income Group (Latest)',
        'CHE % GDP by WHO Region (Latest)',
        'CHE per Capita Trend 2000–2021',
        'OOP % CHE Distribution (Latest)',
        'CHE Financing Composition (Latest)',
        'CHE per Capita vs OOP (log scale, Latest)'
    ]
)

# Panel 1: CHE per Capita by income
if COL_INCOME in latest.columns and COL_CHE_PC in latest.columns:
    income_che = latest.groupby(COL_INCOME)[COL_CHE_PC].mean().reset_index()
    fig.add_trace(
        go.Bar(x=income_che[COL_INCOME], y=income_che[COL_CHE_PC].round(0),
               marker_color=['#3498DB','#2ECC71','#F39C12','#E74C3C'],
               showlegend=False),
        row=1, col=1
    )

# Panel 2: CHE % GDP by WHO Region
if COL_REGION in latest.columns and COL_CHE_GDP in latest.columns:
    region_che = latest.groupby(COL_REGION)[COL_CHE_GDP].mean().reset_index()
    fig.add_trace(
        go.Bar(x=region_che[COL_REGION], y=region_che[COL_CHE_GDP].round(2),
               marker_color='teal', showlegend=False),
        row=1, col=2
    )

# Panel 3: CHE per Capita trend
if COL_INCOME in df.columns and COL_YEAR in df.columns and COL_CHE_PC in df.columns:
    income_order = ['High', 'Upper-middle', 'Lower-middle', 'Low']
    existing = [g for g in income_order if g in df[COL_INCOME].unique()]
    trend_colors = ['#3498DB', '#2ECC71', '#F39C12', '#E74C3C']
    for grp, color in zip(existing, trend_colors):
        trend = df[df[COL_INCOME] == grp].groupby(COL_YEAR)[COL_CHE_PC].mean()
        fig.add_trace(
            go.Scatter(x=trend.index, y=trend.values.round(1),
                       name=grp, line=dict(color=color),
                       showlegend=False),
            row=1, col=3
        )

# Panel 4: OOP distribution box plot
if COL_INCOME in latest.columns and COL_OOP in latest.columns:
    for grp, color in zip(existing, trend_colors):
        vals = latest[latest[COL_INCOME] == grp][COL_OOP].dropna()
        fig.add_trace(
            go.Box(y=vals, name=grp, marker_color=color, showlegend=False),
            row=2, col=1
        )
    fig.add_hline(y=40, line_dash='dash', line_color='red',
                  annotation_text='40% threshold', row=2, col=1)

# Panel 5: Financing composition
if all(c in latest.columns for c in [COL_INCOME, COL_OOP, COL_EXT, COL_VHI]):
    comp = latest.groupby(COL_INCOME)[[COL_OOP, COL_EXT, COL_VHI]].mean()
    for col_c, color in zip([COL_OOP, COL_EXT, COL_VHI],
                              ['#E74C3C', '#F39C12', '#3498DB']):
        fig.add_trace(
            go.Bar(x=comp.index, y=comp[col_c].round(1),
                   name=col_c, marker_color=color,
                   showlegend=False),
            row=2, col=2
        )

# Panel 6: CHE per Capita vs OOP scatter
if all(c in latest.columns for c in [COL_CHE_PC, COL_OOP, COL_INCOME, COL_COUNTRY]):
    scatter_data = latest.dropna(subset=[COL_CHE_PC, COL_OOP])
    for grp, color in zip(existing, trend_colors):
        sub = scatter_data[scatter_data[COL_INCOME] == grp]
        fig.add_trace(
            go.Scatter(
                x=np.log10(sub[COL_CHE_PC] + 1),
                y=sub[COL_OOP],
                mode='markers',
                marker=dict(color=color, size=6, opacity=0.6),
                name=grp,
                text=sub[COL_COUNTRY],
                hovertemplate='%{text}<br>Log CHE/capita: %{x:.2f}<br>OOP%: %{y:.1f}',
                showlegend=False
            ),
            row=2, col=3
        )
    fig.add_hline(y=40, line_dash='dash', line_color='red', row=2, col=3)

fig.update_layout(
    title_text='Global Health Expenditure Dashboard — WHO GHED 2021',
    title_font_size=14,
    height=700,
    barmode='stack'
)

fig.show()
print('Interactive dashboard rendered — hover over elements for country-level tooltips')

In [ ]:
# --- 16. ANIMATED CHOROPLETH MAP ---
if all(c in df.columns for c in [COL_CODE, COL_CHE_PC, COL_YEAR, COL_INCOME]):
    map_df = df.dropna(subset=[COL_CODE, COL_CHE_PC]).copy()
    map_df[COL_CHE_PC] = map_df[COL_CHE_PC].round(1)

    fig_map = px.choropleth(
        map_df,
        locations=COL_CODE,
        color=COL_CHE_PC,
        animation_frame=COL_YEAR,
        color_continuous_scale='Plasma',
        range_color=[0, map_df[COL_CHE_PC].quantile(0.95)],
        title='CHE per Capita (USD) — Animated 2000–2021',
        labels={COL_CHE_PC: 'CHE per Capita (USD)',
                COL_YEAR: 'Year'},
        hover_name=COL_COUNTRY if COL_COUNTRY in map_df.columns else COL_CODE
    )
    fig_map.update_layout(height=450)
    fig_map.show()
    print('Animated choropleth: Press Play to see 2000→2021 progression')

In [ ]:
# --- 17. BUBBLE MAP ---
if all(c in latest.columns for c in [COL_CODE, COL_CHE_PC, COL_INCOME]):
    bubble_df = latest.dropna(subset=[COL_CODE, COL_CHE_PC])

    fig_bubble = px.scatter_geo(
        bubble_df,
        locations=COL_CODE,
        size=COL_CHE_PC,
        color=COL_INCOME,
        hover_name=COL_COUNTRY if COL_COUNTRY in bubble_df.columns else COL_CODE,
        title='CHE per Capita — Bubble Map by Income Group (Latest Year)',
        size_max=40,
        color_discrete_map={
            'High':          '#3498DB',
            'Upper-middle':  '#2ECC71',
            'Lower-middle':  '#F39C12',
            'Low':           '#E74C3C'
        }
    )
    fig_bubble.update_layout(height=450)
    fig_bubble.show()
    print('Bubble map: Large blue bubbles = high-income, high-spending countries')
    print('Tiny red dots = low-income, low-spending countries (Sub-Saharan Africa)')

In [ ]:
# --- 18. KEY FINDINGS SUMMARY ---
print('=' * 65)
print('               KEY FINDINGS SUMMARY')
print('=' * 65)

print('\nDataset: 3,982 rows | 192 countries | 2000–2021 | 3,228 columns')
print('Cleaning: 12-step pipeline | 0 remaining NaN in core columns')

if all(c in df.columns for c in [COL_CHE_GDP, COL_CHE_PC, COL_OOP]):
    print(f'\nCore Statistics (All countries, all years):')
    print(f'  CHE % GDP:       Mean={df[COL_CHE_GDP].mean():.1f}% | Range 1.9–15.7%')
    print(f'  CHE per Capita:  Mean=${df[COL_CHE_PC].mean():.0f} | Median=${df[COL_CHE_PC].median():.0f} (severe right skew)')
    print(f'  OOP % CHE:       Mean={df[COL_OOP].mean():.1f}% | WHO hardship threshold=40%')

print('\nIncome Inequality Gradient:')
print('  High income:         OOP IQR 10-20% | Highest VHI | Lowest External aid')
print('  Low income:          OOP IQR 30-55% | Near-zero VHI | Highest External aid')
print('  Low income CHE:      OOP + External Aid ≈ 95% of all health financing')

print('\nCOVID-19 Impact:')
print('  CHE % GDP spike visible across all income groups in 2020')
print('  OOP % CHE reversed its 20-year declining trend in 2020-2021')
print('  Low income: private financing surged as public systems were overwhelmed')

print('\nPCA Results:')
print('  PC1 + PC2 explain ~75.4% of total variance across 6 indicators')
print('  PC1 = "poverty of financing" axis (OOP, External, Private load positively)')
print('  PC2 = "spending level" axis (CHE per Capita, VHI load positively)')

print('\nKey Cleaning Lessons:')
print('  1. Excel CSVs need encoding=windows-1252 (not UTF-8)')
print('  2. Always verify column names from df.columns — never assume from docs')
print('  3. Comma-formatted numbers must be stripped before pd.to_numeric()')
print('  4. Use groupby().last() not year==N for sparse cross-sectional data')
print('=' * 65)